In [ ]:
import sys, importlib, os
print(sys.executable)
print(sys.path)
print("=== Environment Check / 环境检查 ===\n")
print(f"Python: {sys.version}")
print(f"Executable: {sys.executable}")

all_ok = True
required = ["mujoco", "numpy", "matplotlib", "mediapy"]
for pkg in required:
    try:
        m = importlib.import_module(pkg)
        ver = getattr(m, "__version__", "unknown")
        print(f"  ✓ {pkg:15s} {ver}")
    except ImportError:
        print(f"  ✗ {pkg:15s} NOT FOUND — pip install {pkg}")
        all_ok = False

xml_path = os.path.join("xml_converted", "g1_29dof_mode_11.xml")
mesh_dir = os.path.join("xml_converted", "meshes")
xml_ok = os.path.isfile(xml_path)
n_meshes = len([f for f in os.listdir(mesh_dir) if f.endswith('.STL')]) if os.path.isdir(mesh_dir) else 0
print(f"\nG1 XML:   {'✓ found' if xml_ok else '✗ MISSING'} ({xml_path})")
print(f"Meshes:   {'✓' if n_meshes > 0 else '✗'} {n_meshes} STL files in {mesh_dir}")
all_ok = all_ok and xml_ok and n_meshes > 0

if all_ok:
    print("\n✓ All checks passed. Ready to go!")
    print("✓ 所有检查通过，环境就绪！")
else:
    print("\n⚠ Some checks failed. / 部分检查未通过。")
    print("  Hint: the Jupyter kernel might be using a different Python than where packages are installed.")
    print("  提示：Jupyter kernel 可能使用了与安装包不同的 Python 环境。")
    print(f"  Current kernel Python: {sys.executable}")
    print("  请在 Cursor 右上角切换 kernel 到安装了 mujoco 的 Python 环境。")

# Motion Tracking for Reinforcement Learning in MuJoCo (Unitree G1)
# MuJoCo 强化学习中的动作跟踪（宇树 G1）

This tutorial covers the essential MuJoCo knowledge needed for **motion tracking RL** tasks, using the **Unitree G1 humanoid robot** as the simulation platform. Related methods are commonly seen in works like [DeepMimic](https://xbpeng.github.io/projects/DeepMimic/index.html), [BeyondMimic](https://beyondmimic.github.io/), and various humanoid locomotion papers.
本教程以**宇树 G1 人形机器人**为仿真平台，讲解**动作跟踪强化学习**所需的 MuJoCo 核心知识。相关方法常见于 [DeepMimic](https://xbpeng.github.io/projects/DeepMimic/index.html)、[BeyondMimic](https://beyondmimic.github.io/) 以及各类人形运动控制论文。

Motion tracking RL trains a policy $\pi(a|s)$ to control the Unitree G1 robot so that it **imitates a reference motion** (e.g., from motion capture data).
动作跟踪强化学习训练一个策略 $\pi(a|s)$ 来控制宇树 G1 机器人，使其**模仿参考动作**（例如来自动作捕捉数据）。

The core loop is:
其核心循环如下：

1. At each timestep, the agent observes the **current state** $s_t$ and a **reference target** $s_t^{\text{ref}}$.
1. 在每个时间步，智能体观察**当前状态** $s_t$ 和**参考目标** $s_t^{\text{ref}}$。
2. The agent outputs **actions** $a_t$ (typically target joint angles or torques).
2. 智能体输出**动作** $a_t$（通常是目标关节角或力矩）。
3. The simulator advances one step via physics.
3. 仿真器基于物理规律推进一步。
4. A **tracking reward** $r_t$ measures how closely the simulated pose matches the reference.
4. 用**跟踪奖励** $r_t$ 衡量仿真姿态与参考姿态的接近程度。

This notebook covers the MuJoCo-specific knowledge required at each step:
本 notebook 覆盖每一步所需的 MuJoCo 专项知识：

- Section 1: Unitree G1 model anatomy: bodies, joints, actuators.
- 第 1 节：宇树 G1 模型结构：刚体、关节、执行器。
- Section 2: State representation: `qpos`, `qvel`, and the free joint.
- 第 2 节：状态表示：`qpos`、`qvel` 与 free joint。
- Section 3: Forward kinematics: accessing body positions and orientations.
- 第 3 节：正向运动学：访问刚体位置和姿态。
- Section 4: Quaternion math: MuJoCo orientation utilities.
- 第 4 节：四元数数学：MuJoCo 的姿态工具函数。
- Section 5: Actuator models: torque vs. position control, PD controllers.
- 第 5 节：执行器模型：力矩控制、位置控制与 PD 控制器。
- Section 6: Reference motion: representing and visualizing trajectories.
- 第 6 节：参考动作：轨迹表示与可视化。
- Section 7: Reward design: DeepMimic-style tracking rewards.
- 第 7 节：奖励设计：DeepMimic 风格跟踪奖励。
- Section 8: Complete motion tracking demo.
- 第 8 节：完整动作跟踪示例。
- Section 9: RL integration: observation and action space design.
- 第 9 节：与 RL 对接：观测空间与动作空间设计。



In [ ]:
import mujoco
import mediapy as media
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

## 1. Unitree G1 Model Anatomy
## 1. 宇树 G1 模型结构

The Unitree G1 is a 29-DoF humanoid robot. Its MuJoCo model is defined in MJCF (XML) with mesh geometries (STL files).
宇树 G1 是一款 29 自由度的人形机器人。其 MuJoCo 模型通过 MJCF（XML）定义，使用网格几何体（STL 文件）。

The key components are:
关键组成部分如下：

- **Bodies**: rigid links forming the kinematic tree — pelvis, hip links, knee links, ankle links, waist links, torso, shoulder/elbow/wrist links.
- **Bodies（刚体）**：组成运动学树的刚性连杆——骨盆、髋关节连杆、膝关节连杆、踝关节连杆、腰部连杆、躯干、肩/肘/腕连杆。
- **Joints**: degrees of freedom connecting bodies.
- **Joints（关节）**：连接各刚体的自由度。
- `free`: 6-DoF root joint at the pelvis (3 translation + 3 rotation), stored as 7 values in qpos (3 pos + 4 quat).
- `free`：骨盆处的 6 自由度根关节（3 平移 + 3 旋转），在 qpos 中以 7 个值存储（3 位置 + 4 四元数）。
- `hinge`: 29 revolute joints — legs (6 per side: hip pitch/roll/yaw, knee, ankle pitch/roll), waist (yaw/roll/pitch), arms (7 per side: shoulder pitch/roll/yaw, elbow, wrist roll/pitch/yaw).
- `hinge`：29 个转动关节——腿部（每侧 6 个：髋 pitch/roll/yaw、膝、踝 pitch/roll）、腰部（yaw/roll/pitch）、手臂（每侧 7 个：肩 pitch/roll/yaw、肘、腕 roll/pitch/yaw）。
- **Actuators**: 29 force-limited motor actuators (gear=1), one per hinge joint. Force limits are defined via `actuatorfrcrange` on each joint.
- **Actuators（执行器）**：29 个力矩受限的电机执行器（gear=1），每个 hinge 关节一个。力矩限制通过各关节的 `actuatorfrcrange` 属性定义。
- **Geoms**: mesh geometries (STL) for visual rendering and collision detection.
- **Geoms（几何体）**：用于可视化渲染和碰撞检测的网格几何体（STL）。

Below we load the G1 model from `xml_converted/g1_29dof_mode_11.xml` and inspect its structure.
下面我们从 `xml_converted/g1_29dof_mode_11.xml` 加载 G1 模型并查看其结构。

In [ ]:
import os

G1_XML_PATH = os.path.join('xml_converted', 'g1_29dof_mode_11.xml')
os.makedirs("imgs", exist_ok=True)

model = mujoco.MjModel.from_xml_path(G1_XML_PATH)
data = mujoco.MjData(model)

print(f"Model: Unitree G1 29-DoF")
print(f"Degrees of freedom (nv): {model.nv}")
print(f"Generalized coordinates (nq): {model.nq}")
print(f"Actuators (nu): {model.nu}")
print(f"Bodies (nbody): {model.nbody}")
print(f"Joints (njnt): {model.njnt}")
print()
print("--- Joint Layout ---")
type_names = {0: 'free', 1: 'ball', 2: 'slide', 3: 'hinge'}
qpos_sizes = {0: 7, 1: 4, 2: 1, 3: 1}
qpos_offset = 0
for i in range(model.njnt):
    jtype = model.jnt_type[i]
    nq = qpos_sizes[jtype]
    print(f"  joint {i:2d}: {model.joint(i).name:35s} type={type_names[jtype]:5s}  qpos[{qpos_offset}:{qpos_offset+nq}]")
    qpos_offset += nq

In [ ]:
# Visualize the humanoid in its initial pose
mujoco.mj_resetData(model, data)
mujoco.mj_forward(model, data)

renderer = mujoco.Renderer(model, height=480, width=640)
renderer.update_scene(data)
media.show_image(renderer.render())

## 2. State Representation: `qpos`, `qvel` and the Free Joint
## 2. 状态表示：`qpos`、`qvel` 与 Free Joint

In motion tracking RL, the **state** of the character is fully described by two vectors:
在动作跟踪强化学习中，角色的**状态**可由两个向量完整描述：

- `data.qpos`: Generalized positions, dimension `model.nq`.
- `data.qpos`：广义位置，维度为 `model.nq`。
- `data.qvel`: Generalized velocities, dimension `model.nv`.
- `data.qvel`：广义速度，维度为 `model.nv`。

**Key insight**: `nq != nv` because the free joint uses a **quaternion** (4 values) for orientation in `qpos`, but only **3 angular velocity** values in `qvel`.
**关键点**：`nq != nv`，因为 free joint 在 `qpos` 中用 **四元数**（4 个值）表示姿态，而在 `qvel` 中仅用 **3 个角速度**值。

For the Unitree G1:
对于宇树 G1：

- `qpos` has 36 dimensions: 7 (free joint: 3 pos + 4 quat) + 29 (hinge joints: 1 each).
- `qpos` 共有 36 维：7（free joint：3 位置 + 4 四元数）+ 29（hinge 关节：每个 1 维）。
- `qvel` has 35 dimensions: 6 (free joint: 3 linear vel + 3 angular vel) + 29 (hinge joints: 1 each).
- `qvel` 共有 35 维：6（free joint：3 线速度 + 3 角速度）+ 29（hinge 关节：每个 1 维）。

### Free joint qpos layout
### Free joint 的 qpos 布局

`qpos[0:3]  = (x, y, z)`: root position in world frame.
`qpos[0:3]  = (x, y, z)`：世界坐标系下的根位置。

`qpos[3:7]  = (w, x, y, z)`: root orientation as unit quaternion (MuJoCo convention: w first).
`qpos[3:7]  = (w, x, y, z)`：根姿态的单位四元数（MuJoCo 约定：w 在前）。

`qpos[7:]   = joint angles`: one per hinge joint (radians).
`qpos[7:]   = joint angles`：各 hinge 关节角（弧度）。

### Free joint qvel layout
### Free joint 的 qvel 布局

`qvel[0:3]  = (vx, vy, vz)`: root linear velocity in world frame.
`qvel[0:3]  = (vx, vy, vz)`：世界坐标系下的根线速度。

`qvel[3:6]  = (wx, wy, wz)`: root angular velocity in world frame.
`qvel[3:6]  = (wx, wy, wz)`：世界坐标系下的根角速度。

`qvel[6:]   = joint velocities`: one per hinge joint (rad/s).
`qvel[6:]   = joint velocities`：各 hinge 关节角速度（rad/s）。



In [ ]:
mujoco.mj_resetData(model, data)
mujoco.mj_forward(model, data)

print(f"qpos shape: {data.qpos.shape}  (nq={model.nq})")
print(f"qvel shape: {data.qvel.shape}  (nv={model.nv})")
print()

# Decompose qpos
root_pos = data.qpos[:3]
root_quat = data.qpos[3:7]  # (w, x, y, z)
joint_angles = data.qpos[7:]

print(f"Root position:    {root_pos}")
print(f"Root quaternion:  {root_quat}  (identity = [1,0,0,0])")
print(f"Joint angles:     {joint_angles}  (all zeros at reset)")
print()

# Decompose qvel
root_lin_vel = data.qvel[:3]
root_ang_vel = data.qvel[3:6]
joint_vels = data.qvel[6:]

print(f"Root linear vel:  {root_lin_vel}")
print(f"Root angular vel: {root_ang_vel}")
print(f"Joint velocities: {joint_vels}")

In [ ]:
mujoco.mj_resetData(model, data)

right_knee_jnt_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, 'right_knee_joint')
right_knee_qpos_adr = model.jnt_qposadr[right_knee_jnt_id]
print(f"right_knee_joint id: {right_knee_jnt_id}, qpos address: {right_knee_qpos_adr}")

data.qpos[right_knee_qpos_adr] = 1.0  # radians (G1 XML uses radian convention)

rs1_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, 'right_shoulder_pitch_joint')
data.qpos[model.jnt_qposadr[rs1_id]] = 0.7

mujoco.mj_forward(model, data)
renderer.update_scene(data)
media.show_image(renderer.render())

## 3. Forward Kinematics: Body Positions & Orientations
## 3. 正向运动学：刚体位置与姿态

After calling `mj_forward(model, data)` or `mj_step(model, data)`, MuJoCo computes all **derived quantities** from the current state.
调用 `mj_forward(model, data)` 或 `mj_step(model, data)` 后，MuJoCo 会根据当前状态计算所有**派生量**。

For motion tracking, the most important fields are:
对于动作跟踪任务，最重要的字段如下：

- `data.xpos[body_id]`: body position in world frame, shape `(3,)`.
- `data.xpos[body_id]`：刚体在世界坐标系中的位置，形状为 `(3,)`。
- `data.xquat[body_id]`: body orientation quaternion, shape `(4,)` in `(w,x,y,z)`.
- `data.xquat[body_id]`：刚体姿态四元数，形状为 `(4,)`，顺序为 `(w,x,y,z)`。
- `data.xmat[body_id]`: body orientation as flattened 3x3 rotation matrix, shape `(9,)`.
- `data.xmat[body_id]`：刚体姿态的 3x3 旋转矩阵（展平后），形状为 `(9,)`。
- `data.cvel[body_id]`: body 6D velocity (angular, linear) in world frame, shape `(6,)`.
- `data.cvel[body_id]`：刚体在世界坐标系中的 6 维速度（角速度、线速度），形状为 `(6,)`。
- `data.subtree_com[body_id]`: center of mass of the subtree rooted at this body, shape `(3,)`.
- `data.subtree_com[body_id]`：以该刚体为根的子树质心，形状为 `(3,)`。
- `data.site_xpos[site_id]`: site position in world frame, shape `(3,)`. G1 has IMU sites: `imu_in_torso`, `imu_in_pelvis`.
- `data.site_xpos[site_id]`：site 在世界坐标系中的位置，形状为 `(3,)`。G1 提供 IMU 站点：`imu_in_torso`、`imu_in_pelvis`。

These quantities are used in reward computation, such as comparing simulated and reference body positions or orientations. For the G1, end-effector tracking uses body positions (hands: `left/right_wrist_yaw_link`, feet: `left/right_ankle_roll_link`).
这些量会用于奖励计算，例如比较仿真与参考的刚体位置或姿态差异。对于 G1，末端执行器跟踪使用刚体位置（手：`left/right_wrist_yaw_link`，脚：`left/right_ankle_roll_link`）。



In [ ]:
mujoco.mj_resetData(model, data)
mujoco.mj_forward(model, data)

print("--- Body positions (xpos) ---")
for i in range(model.nbody):
    name = model.body(i).name
    pos = data.xpos[i]
    quat = data.xquat[i]
    print(f"  {name:35s}  pos={pos}  quat={quat}")

print()
print("--- End-effector bodies (hands & feet) ---")
for name in ['left_wrist_yaw_link', 'right_wrist_yaw_link', 'left_ankle_roll_link', 'right_ankle_roll_link']:
    bid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, name)
    print(f"  {name}: pos={data.xpos[bid]}")

print()
print("--- IMU sites ---")
for name in ['imu_in_torso', 'imu_in_pelvis']:
    sid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, name)
    print(f"  {name}: pos={data.site_xpos[sid]}")

print(f"\n--- Center of mass ---")
print(f"  Whole-body CoM: {data.subtree_com[0]}")

## 4. Quaternion Math in MuJoCo
## 4. MuJoCo 中的四元数数学

Orientation tracking is at the heart of motion imitation.
姿态跟踪是动作模仿任务的核心。

MuJoCo uses quaternions in `(w, x, y, z)` convention and provides utility functions:
MuJoCo 使用 `(w, x, y, z)` 约定的四元数，并提供如下工具函数：

- `mju_mulQuat(res, q1, q2)`: quaternion multiplication, `res = q1 * q2`.
- `mju_mulQuat(res, q1, q2)`：四元数乘法，`res = q1 * q2`。
- `mju_negQuat(res, q)`: quaternion conjugate/inverse, `res = q*`.
- `mju_negQuat(res, q)`：四元数共轭/逆，`res = q*`。
- `mju_quat2Vel(vel, q, dt)`: convert quaternion difference to angular velocity.
- `mju_quat2Vel(vel, q, dt)`：将四元数差转换为角速度。
- `mju_mat2Quat(quat, mat)`: convert 3x3 rotation matrix to quaternion.
- `mju_mat2Quat(quat, mat)`：将 3x3 旋转矩阵转换为四元数。
- `mju_quat2Mat(mat, quat)`: convert quaternion to 3x3 rotation matrix.
- `mju_quat2Mat(mat, quat)`：将四元数转换为 3x3 旋转矩阵。
- `mju_axisAngle2Quat(quat, axis, angle)`: axis-angle to quaternion.
- `mju_axisAngle2Quat(quat, axis, angle)`：轴角表示转四元数。
- `mju_subQuat(res, qa, qb)`: compute rotation from `qb` to `qa`.
- `mju_subQuat(res, qa, qb)`：计算从 `qb` 到 `qa` 的旋转差。

### Computing orientation error
### 计算姿态误差

To measure orientation difference between simulation and reference, compute:
要衡量仿真姿态与参考姿态的差异，可计算：

$$q_{\text{error}} = q_{\text{sim}} \otimes q_{\text{ref}}^{-1}$$
$$q_{\text{error}} = q_{\text{sim}} \otimes q_{\text{ref}}^{-1}$$

Then convert it to an angular velocity vector via `mju_quat2Vel`.
然后通过 `mju_quat2Vel` 将其转换为角速度向量。

The norm of this vector gives orientation error in radians.
该向量的范数即为以弧度表示的姿态误差。



In [ ]:
def quat_error(q_sim, q_ref):
    """Compute orientation error between two quaternions as an angular velocity vector.
    Returns a 3D vector whose norm is the angle of rotation (in radians) between the two orientations.
    """
    q_ref_conj = np.zeros(4)
    q_err = np.zeros(4)
    vel = np.zeros(3)
    
    mujoco.mju_negQuat(q_ref_conj, q_ref)       # q_ref_conj = q_ref^{-1}
    mujoco.mju_mulQuat(q_err, q_sim, q_ref_conj) # q_err = q_sim * q_ref^{-1}
    mujoco.mju_quat2Vel(vel, q_err, 1.0)         # convert to angular velocity (dt=1 -> gives angle directly)
    return vel

# Example: identity vs 90-degree rotation around Z
q_identity = np.array([1.0, 0.0, 0.0, 0.0])
q_90z = np.zeros(4)
mujoco.mju_axisAngle2Quat(q_90z, np.array([0.0, 0.0, 1.0]), np.pi/2)

err = quat_error(q_90z, q_identity)
print(f"q_identity:   {q_identity}")
print(f"q_90z:        {q_90z}")
print(f"Error vector: {err}")
print(f"Error angle:  {np.linalg.norm(err):.4f} rad = {np.degrees(np.linalg.norm(err)):.1f} deg")

# Verify: same quaternion -> zero error
err_zero = quat_error(q_identity, q_identity)
print(f"\nSame quat error: {err_zero}  (should be zeros)")

# mju_subQuat: an alternative that directly gives the angular difference
sub_res = np.zeros(3)
mujoco.mju_subQuat(sub_res, q_90z, q_identity)
print(f"mju_subQuat result: {sub_res}  (should match error vector above)")

## 5. Actuator Models: Torque Control vs. Position Control
## 5. 执行器模型：力矩控制 vs 位置控制

In motion tracking RL, the agent's action is typically one of the following:
在动作跟踪强化学习中，智能体动作通常采用以下形式之一：

1. **Torque control** (`motor` actuator): action is a torque applied directly. The G1 uses this model with `gear=1`, so `data.ctrl[i]` is the torque in N·m.
1. **力矩控制**（`motor` 执行器）：动作直接施加力矩。G1 使用该模式，`gear=1`，因此 `data.ctrl[i]` 就是力矩（单位 N·m）。

2. **Position control** (`position` actuator): action is target joint angle with a built-in PD controller.
2. **位置控制**（`position` 执行器）：动作是目标关节角，由内置 PD 控制器跟踪。

3. **PD target** (common in DeepMimic/BeyondMimic): policy outputs target joint angles $q^{\text{target}}$, and PD computes torques.
3. **PD 目标控制**（DeepMimic/BeyondMimic 常用）：策略输出目标关节角 $q^{\text{target}}$，由 PD 计算力矩。

$$\tau = k_p (q^{\text{target}} - q) - k_d \dot{q}$$
$$\tau = k_p (q^{\text{target}} - q) - k_d \dot{q}$$

The G1 model uses motor actuators with `gear=1` and force limits via `actuatorfrcrange` on each joint. We apply PD control on top.
G1 模型使用 `gear=1` 的 motor 执行器，每个关节通过 `actuatorfrcrange` 限制力矩。我们在其上实现 PD 控制器。

In [ ]:
print("--- Actuator info ---")
for i in range(model.nu):
    name = model.actuator(i).name
    gear = model.actuator_gear[i, 0]
    jnt_id = model.actuator_trnid[i, 0]
    jnt_range = model.jnt_range[jnt_id]
    frc_range = model.jnt_actfrcrange[jnt_id]
    print(f"  {name:35s}  gear={gear:4.1f}  range=[{jnt_range[0]:7.3f}, {jnt_range[1]:6.3f}]  frc=[{frc_range[0]:7.0f}, {frc_range[1]:5.0f}]")

In [ ]:
def pd_controller(model, data, target_qpos, kp, kd):
    """Compute torques using a PD controller.
    target_qpos: desired joint angles (excluding free joint), shape (nu,)
    Returns: torques shape (nu,)
    """
    current_qpos = data.qpos[7:]
    current_qvel = data.qvel[6:]
    position_error = target_qpos - current_qpos
    torques = kp * position_error - kd * current_qvel
    return torques

def get_force_limits(model):
    """Extract per-actuator force limits from joint actuatorfrcrange."""
    limits = np.full(model.nu, 200.0)
    try:
        for i in range(model.nu):
            jnt_id = model.actuator_trnid[i, 0]
            limits[i] = abs(model.jnt_actfrcrange[jnt_id, 1])
    except (AttributeError, IndexError):
        pass
    return limits

n_joints = model.nu
force_limits = get_force_limits(model)
kp = np.full(n_joints, 200.0)
kd = np.full(n_joints, 20.0)

mujoco.mj_resetData(model, data)
target = data.qpos[7:].copy()

frames = []
for step in range(500):
    torques = pd_controller(model, data, target, kp, kd)
    data.ctrl[:] = np.clip(torques, -force_limits, force_limits)
    
    mujoco.mj_step(model, data)
    if step % 10 == 0:
        renderer.update_scene(data)
        frames.append(renderer.render().copy())

media.show_video(frames, fps=25, title='PD controller holding initial pose')

## 6. Reference Motion: Representing & Visualizing Trajectories
## 6. 参考动作：轨迹表示与可视化

In motion tracking RL, a reference motion is a time-indexed sequence of target poses:
在动作跟踪强化学习中，参考动作是按时间索引的目标姿态序列：

$$\{(q^{\text{ref}}_t, \dot{q}^{\text{ref}}_t)\}_{t=0}^{T}$$
$$\{(q^{\text{ref}}_t, \dot{q}^{\text{ref}}_t)\}_{t=0}^{T}$$

In practice, it can come from:
在实践中，它通常来自：

- Motion capture (MoCap) data retargeted to the simulation model.
- 重定向到仿真模型的动作捕捉（MoCap）数据。
- Kinematic trajectory optimization.
- 运动学轨迹优化结果。
- Hand-crafted keyframes for simple motions.
- 手工设计的关键帧（适用于简单动作）。

Each frame contains a full `qpos` vector.
每一帧都包含完整的 `qpos` 向量。

For the free joint, this includes root position and root orientation.
对于 free joint，这其中包含根位置与根姿态。

Below we generate a simple walking-like reference motion via keyframe interpolation for demonstration.
下面我们通过关键帧插值生成一个简化的类行走参考动作来演示这个概念。



In [ ]:
def get_joint_qpos_idx(model, joint_name):
    """Get the qpos index for a named joint."""
    jnt_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, joint_name)
    return model.jnt_qposadr[jnt_id]

def create_reference_motion(model, duration=2.0, dt=0.01):
    """Generate a simple cyclic leg-swing reference motion for the G1.
    Returns: times (T,), ref_qpos (T, nq)
    """
    T = int(duration / dt)
    times = np.arange(T) * dt
    ref_qpos = np.zeros((T, model.nq))
    
    base_qpos = np.zeros(model.nq)
    base_qpos[2] = 0.793    # G1 pelvis height
    base_qpos[3] = 1.0      # quaternion w (identity)
    
    freq = 2.0 * np.pi / duration
    
    rhp = get_joint_qpos_idx(model, 'right_hip_pitch_joint')
    rk  = get_joint_qpos_idx(model, 'right_knee_joint')
    rap = get_joint_qpos_idx(model, 'right_ankle_pitch_joint')
    lhp = get_joint_qpos_idx(model, 'left_hip_pitch_joint')
    lk  = get_joint_qpos_idx(model, 'left_knee_joint')
    lap = get_joint_qpos_idx(model, 'left_ankle_pitch_joint')
    rsp = get_joint_qpos_idx(model, 'right_shoulder_pitch_joint')
    lsp = get_joint_qpos_idx(model, 'left_shoulder_pitch_joint')
    
    for i, t in enumerate(times):
        qpos = base_qpos.copy()
        phase = freq * t
        
        qpos[0] = 0.3 * t  # slow forward motion
        
        hip_amp = 0.5  # ~29 deg
        qpos[rhp] = -hip_amp * np.sin(phase)
        qpos[lhp] = hip_amp * np.sin(phase)
        
        knee_amp = 0.7
        qpos[rk] = knee_amp * np.maximum(0, np.sin(phase))
        qpos[lk] = knee_amp * np.maximum(0, -np.sin(phase))
        
        ankle_amp = 0.17  # ~10 deg
        qpos[rap] = ankle_amp * np.sin(phase)
        qpos[lap] = -ankle_amp * np.sin(phase)
        
        arm_amp = 0.35  # ~20 deg
        qpos[rsp] = arm_amp * np.sin(phase)
        qpos[lsp] = -arm_amp * np.sin(phase)
        
        ref_qpos[i] = qpos
    
    return times, ref_qpos

ref_times, ref_qpos = create_reference_motion(model, duration=2.0, dt=0.01)
print(f"Reference motion: {len(ref_times)} frames, duration={ref_times[-1]:.2f}s")
print(f"qpos shape per frame: {ref_qpos[0].shape}")

In [ ]:
# Visualize the reference motion by directly setting qpos
frames_ref = []
for i in range(0, len(ref_qpos), 5):  # every 5th frame
    data.qpos[:] = ref_qpos[i]
    mujoco.mj_forward(model, data)
    renderer.update_scene(data)
    frames_ref.append(renderer.render().copy())

media.show_video(frames_ref, fps=20, title='Reference motion (kinematic playback)')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

joints_to_plot = [
    ('right_hip_pitch_joint', 'Right Hip Pitch'),
    ('right_knee_joint', 'Right Knee'),
    ('left_hip_pitch_joint', 'Left Hip Pitch'),
    ('left_knee_joint', 'Left Knee'),
]

for ax, (jname, label) in zip(axes.flat, joints_to_plot):
    idx = get_joint_qpos_idx(model, jname)
    ax.plot(ref_times, np.degrees(ref_qpos[:, idx]))
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Angle (deg)')
    ax.set_title(label)
    ax.grid(True, alpha=0.3)

plt.suptitle('Reference Motion Joint Trajectories (Unitree G1)', fontsize=14)
plt.tight_layout()
plt.savefig('imgs/ref_motion_trajectories.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Reward Design: DeepMimic-style Tracking Rewards
## 7. 奖励设计：DeepMimic 风格跟踪奖励

The reward function is the core of motion tracking RL.
奖励函数是动作跟踪强化学习的核心。

Following DeepMimic, the total reward is a weighted sum of tracking terms:
按照 DeepMimic，总代价通常写成多个跟踪项的加权和：

$$r_t = w_q \cdot r_q + w_v \cdot r_v + w_e \cdot r_e + w_c \cdot r_c$$
$$r_t = w_q \cdot r_q + w_v \cdot r_v + w_e \cdot r_e + w_c \cdot r_c$$

- $r_q = \exp\left(-k_q \sum_j \|\hat{q}_j \ominus q_j^{\text{ref}}\|^2\right)$ tracks joint orientations.
- $r_q = \exp\left(-k_q \sum_j \|\hat{q}_j \ominus q_j^{\text{ref}}\|^2\right)$：跟踪关节姿态。
- $r_v = \exp\left(-k_v \sum_j \|\dot{q}_j - \dot{q}_j^{\text{ref}}\|^2\right)$ tracks joint velocities.
- $r_v = \exp\left(-k_v \sum_j \|\dot{q}_j - \dot{q}_j^{\text{ref}}\|^2\right)$：跟踪关节速度。
- $r_e = \exp\left(-k_e \sum_e \|p_e - p_e^{\text{ref}}\|^2\right)$ tracks end-effector positions.
- $r_e = \exp\left(-k_e \sum_e \|p_e - p_e^{\text{ref}}\|^2\right)$：跟踪末端执行器位置。
- $r_c = \exp\left(-k_c \|p_{\text{com}} - p_{\text{com}}^{\text{ref}}\|^2\right)$ tracks center of mass.
- $r_c = \exp\left(-k_c \|p_{\text{com}} - p_{\text{com}}^{\text{ref}}\|^2\right)$：跟踪质心位置。

The $\exp(-k \cdot \text{error}^2)$ form keeps rewards in $[0,1]$ with smooth decay.
$\exp(-k \cdot \text{error}^2)$ 的形式能让奖励保持在 $[0,1]$ 范围内并平滑衰减。



In [ ]:
class MotionTrackingReward:
    """DeepMimic-style motion tracking reward computation for the Unitree G1."""
    
    def __init__(self, model,
                 w_qpos=0.5, w_qvel=0.1, w_end_effector=0.15, w_com=0.1, w_root_orient=0.15,
                 k_qpos=2.0, k_qvel=0.1, k_end_effector=40.0, k_com=10.0, k_root_orient=5.0,
                 end_effector_bodies=None):
        self.model = model
        
        self.w_qpos = w_qpos
        self.w_qvel = w_qvel
        self.w_ee = w_end_effector
        self.w_com = w_com
        self.w_root = w_root_orient
        
        self.k_qpos = k_qpos
        self.k_qvel = k_qvel
        self.k_ee = k_end_effector
        self.k_com = k_com
        self.k_root = k_root_orient
        
        if end_effector_bodies is None:
            end_effector_bodies = [
                'left_wrist_yaw_link', 'right_wrist_yaw_link',
                'left_ankle_roll_link', 'right_ankle_roll_link'
            ]
        self.ee_body_ids = [
            mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, name)
            for name in end_effector_bodies
        ]
        self.root_body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, 'pelvis')
    
    def compute(self, data, ref_qpos, ref_qvel, ref_data):
        rewards = {}
        
        joint_diff = data.qpos[7:] - ref_qpos[7:]
        rewards['qpos'] = np.exp(-self.k_qpos * np.sum(joint_diff ** 2))
        
        vel_diff = data.qvel[6:] - ref_qvel[6:]
        rewards['qvel'] = np.exp(-self.k_qvel * np.sum(vel_diff ** 2))
        
        ee_error = 0.0
        for bid in self.ee_body_ids:
            diff = data.xpos[bid] - ref_data.xpos[bid]
            ee_error += np.sum(diff ** 2)
        rewards['end_effector'] = np.exp(-self.k_ee * ee_error)
        
        com_diff = data.subtree_com[0] - ref_data.subtree_com[0]
        rewards['com'] = np.exp(-self.k_com * np.sum(com_diff ** 2))
        
        root_ori_err = quat_error(data.xquat[self.root_body_id], ref_data.xquat[self.root_body_id])
        rewards['root_orient'] = np.exp(-self.k_root * np.sum(root_ori_err ** 2))
        
        total = (self.w_qpos * rewards['qpos'] +
                 self.w_qvel * rewards['qvel'] +
                 self.w_ee * rewards['end_effector'] +
                 self.w_com * rewards['com'] +
                 self.w_root * rewards['root_orient'])
        
        return total, rewards

reward_fn = MotionTrackingReward(model)
print("Reward function initialized.")
print(f"Weights: qpos={reward_fn.w_qpos}, qvel={reward_fn.w_qvel}, "
      f"ee={reward_fn.w_ee}, com={reward_fn.w_com}, root={reward_fn.w_root}")

In [ ]:
# Demonstrate the reward function
# Case 1: Perfect tracking (sim = ref) -> reward should be ~1.0
ref_data = mujoco.MjData(model)

# Set both to the same pose
test_pose = ref_qpos[50]  # pick a frame from reference
data.qpos[:] = test_pose
ref_data.qpos[:] = test_pose
data.qvel[:] = 0
ref_data.qvel[:] = 0

mujoco.mj_forward(model, data)
mujoco.mj_forward(model, ref_data)

total, components = reward_fn.compute(data, test_pose, np.zeros(model.nv), ref_data)
print("Case 1: Perfect tracking")
print(f"  Total reward: {total:.4f}")
for k, v in components.items():
    print(f"  {k:15s}: {v:.4f}")

print()

# Case 2: Moderate error
noisy_pose = test_pose.copy()
noisy_pose[7:] += np.random.randn(model.nq - 7) * 0.2  # add noise to joint angles
data.qpos[:] = noisy_pose
mujoco.mj_forward(model, data)

total2, components2 = reward_fn.compute(data, test_pose, np.zeros(model.nv), ref_data)
print("Case 2: Noisy joints (std=0.2 rad)")
print(f"  Total reward: {total2:.4f}")
for k, v in components2.items():
    print(f"  {k:15s}: {v:.4f}")

In [ ]:
# Visualize how reward decreases as error increases
errors = np.linspace(0, 2, 100)

fig, ax = plt.subplots(figsize=(8, 5))
for k, label in [(0.5, 'k=0.5 (loose)'), (2.0, 'k=2.0 (default)'), 
                  (5.0, 'k=5.0 (tight)'), (10.0, 'k=10.0 (very tight)')]:
    ax.plot(errors, np.exp(-k * errors**2), label=label)

ax.set_xlabel('Error (radians or meters)')
ax.set_ylabel('Reward')
ax.set_title('exp(-k * error²): Tracking Reward vs Error')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('imgs/reward_curves.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Complete Motion Tracking Demo
## 8. 完整动作跟踪示例

Now we put everything together into a complete motion tracking loop:
现在我们把前面的内容整合成完整的动作跟踪循环：

1. Load reference motion.
1. 加载参考动作。
2. At each timestep, look up current reference pose.
2. 在每个时间步查询当前参考姿态。
3. Use PD control to track reference joint angles (simulating RL policy output).
3. 使用 PD 控制跟踪参考关节角（模拟 RL 策略输出）。
4. Step the simulation.
4. 推进一步仿真。
5. Compute tracking reward.
5. 计算跟踪奖励。
6. Render and log results.
6. 渲染并记录结果。

This is the inner loop that would later be wrapped by an RL training framework.
这就是后续可封装进 RL 训练框架的“内循环”。



In [ ]:
def run_motion_tracking(model, ref_times, ref_qpos, kp=200.0, kd=20.0, render_every=10):
    """Run a complete motion tracking simulation with PD control."""
    data = mujoco.MjData(model)
    ref_data = mujoco.MjData(model)
    renderer_local = mujoco.Renderer(model, height=480, width=640)
    
    sim_dt = model.opt.timestep
    ref_dt = ref_times[1] - ref_times[0]
    steps_per_ref = max(1, int(ref_dt / sim_dt))
    
    reward_fn_local = MotionTrackingReward(model)
    
    data.qpos[:] = ref_qpos[0]
    data.qvel[:] = 0
    mujoco.mj_forward(model, data)
    
    kp_arr = np.full(model.nu, kp)
    kd_arr = np.full(model.nu, kd)
    frc_limits = get_force_limits(model)
    
    frames = []
    rewards_log = []
    component_log = {k: [] for k in ['qpos', 'qvel', 'end_effector', 'com', 'root_orient']}
    
    total_steps = 0
    for ref_idx in range(len(ref_qpos) - 1):
        ref_qvel = np.zeros(model.nv)
        ref_qvel[6:] = (ref_qpos[ref_idx + 1, 7:] - ref_qpos[ref_idx, 7:]) / ref_dt
        
        ref_data.qpos[:] = ref_qpos[ref_idx]
        mujoco.mj_forward(model, ref_data)
        
        target_joints = ref_qpos[ref_idx, 7:]
        
        for sub_step in range(steps_per_ref):
            torques = pd_controller(model, data, target_joints, kp_arr, kd_arr)
            data.ctrl[:] = np.clip(torques, -frc_limits, frc_limits)
            
            mujoco.mj_step(model, data)
            total_steps += 1
            
            if total_steps % render_every == 0:
                renderer_local.update_scene(data)
                frames.append(renderer_local.render().copy())
        
        total_r, comp = reward_fn_local.compute(data, ref_qpos[ref_idx], ref_qvel, ref_data)
        rewards_log.append(total_r)
        for k in component_log:
            component_log[k].append(comp[k])
    
    renderer_local.close()
    return frames, np.array(rewards_log), {k: np.array(v) for k, v in component_log.items()}

print("Running motion tracking simulation...")
frames_track, rewards, components = run_motion_tracking(
    model, ref_times, ref_qpos, kp=300.0, kd=30.0, render_every=20
)
print(f"Done! {len(frames_track)} frames, avg reward: {rewards.mean():.4f}")

In [ ]:
media.show_video(frames_track, fps=25, title='PD tracking of reference motion')

In [ ]:
# Plot tracking rewards over time
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Total reward
axes[0].plot(ref_times[:len(rewards)], rewards)
axes[0].set_ylabel('Total Reward')
axes[0].set_title('Motion Tracking Reward Over Time')
axes[0].set_ylim(0, 1.1)
axes[0].grid(True, alpha=0.3)

# Component rewards
for name, values in components.items():
    axes[1].plot(ref_times[:len(values)], values, label=name)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Reward Component')
axes[1].set_title('Individual Reward Components')
axes[1].set_ylim(0, 1.1)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('imgs/tracking_rewards.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
data_sim = mujoco.MjData(model)
data_ref = mujoco.MjData(model)
renderer_sim = mujoco.Renderer(model, height=360, width=480)
renderer_ref = mujoco.Renderer(model, height=360, width=480)

data_sim.qpos[:] = ref_qpos[0]
data_sim.qvel[:] = 0
mujoco.mj_forward(model, data_sim)

kp_arr = np.full(model.nu, 300.0)
kd_arr = np.full(model.nu, 30.0)
frc_limits = get_force_limits(model)

ref_dt = ref_times[1] - ref_times[0]
sim_dt = model.opt.timestep
steps_per_ref = max(1, int(ref_dt / sim_dt))

comparison_frames = []
for ref_idx in range(0, len(ref_qpos) - 1, 3):
    target = ref_qpos[ref_idx, 7:]
    for _ in range(steps_per_ref * 3):
        torques = pd_controller(model, data_sim, target, kp_arr, kd_arr)
        data_sim.ctrl[:] = np.clip(torques, -frc_limits, frc_limits)
        mujoco.mj_step(model, data_sim)
    
    renderer_sim.update_scene(data_sim)
    img_sim = renderer_sim.render().copy()
    
    data_ref.qpos[:] = ref_qpos[ref_idx]
    mujoco.mj_forward(model, data_ref)
    renderer_ref.update_scene(data_ref)
    img_ref = renderer_ref.render().copy()
    
    combined = np.concatenate([img_ref, img_sim], axis=1)
    comparison_frames.append(combined)

renderer_sim.close()
renderer_ref.close()

media.show_video(comparison_frames, fps=15, title='Left: Reference (kinematic) | Right: PD Tracked (physics)')

## 9. RL Integration: Observation, Action, and Episode Design
## 9. RL 对接：观测、动作与回合设计

When wrapping this into an RL environment (e.g., Gymnasium), key decisions include observation, action, and episode design.
当把上述系统封装为 RL 环境（如 Gymnasium）时，关键设计包括观测、动作与回合机制。

### Observation Space
### 观测空间

Typical observation components are:
典型观测通常包含：

- Root height: `data.qpos[2]` (G1 pelvis ~0.793m standing).
- 根高度：`data.qpos[2]`（G1 站立时骨盆约 0.793m）。
- Root orientation: `data.qpos[3:7]` (quaternion).
- 根姿态：`data.qpos[3:7]`（四元数）。
- Joint angles: `data.qpos[7:]`.
- 关节角：`data.qpos[7:]`。
- Root velocity: `data.qvel[0:6]` (linear + angular).
- 根速度：`data.qvel[0:6]`（线速度 + 角速度）。
- Joint velocities: `data.qvel[6:]`.
- 关节速度：`data.qvel[6:]`。
- Reference target: relative reference pose at current phase.
- 参考目标：当前相位对应的参考姿态（通常是相对量）。
- Phase variable: position in the reference motion cycle.
- 相位变量：在参考动作周期中的位置。

Important: the reference target is often represented as pose difference or direct reference pose, and look-ahead future reference frames can help.
重要说明：参考目标常表示为姿态差值或直接参考姿态，引入前瞻参考帧通常也有帮助。

### Action Space
### 动作空间

- PD targets: target joint angles, then PD computes torques.
- PD 目标：输出目标关节角，再由 PD 计算力矩。
- Residual PD: action is an offset added to reference angles.
- 残差 PD：动作是加在参考角度上的偏移量。
- Direct torque: normalized torques applied directly.
- 直接力矩：直接施加归一化力矩。

Residual PD is popular: `target = ref_qpos[7:] + action * scale`.
残差 PD 很常见：`target = ref_qpos[7:] + action * scale`。

The policy learns corrections to the reference rather than generating motion from scratch.
策略学习的是对参考动作的修正，而不是从零生成动作。

### Episode Design
### 回合设计

- Reference State Initialization (RSI): randomly sample a starting frame from reference motion and initialize qpos/qvel accordingly.
- 参考状态初始化（RSI）：从参考动作中随机采样起始帧，并据此初始化 qpos/qvel。

This is crucial for learning; without RSI, policy may only learn to track from the beginning.
这对学习至关重要；没有 RSI，策略往往只能学会从开头开始跟踪。

- Early termination: end episode if robot falls (e.g., pelvis height below 0.3m).
- 提前终止：若机器人跌倒（如骨盆高度低于 0.3m）则结束回合。
- Cyclic motions: for locomotion, reference motion loops and phase tracks cycle position.
- 周期动作：对行走等任务，参考动作会循环，相位变量用于标记周期位置。



In [ ]:
class MotionTrackingEnv:
    """Minimal motion tracking RL environment for the Unitree G1."""
    
    def __init__(self, model_xml_path, ref_qpos, ref_times,
                 kp=300.0, kd=30.0, action_scale=0.3, ctrl_dt=0.02):
        self.model = mujoco.MjModel.from_xml_path(model_xml_path)
        self.data = mujoco.MjData(self.model)
        self.ref_data = mujoco.MjData(self.model)
        
        self.ref_qpos = ref_qpos
        self.ref_times = ref_times
        self.ref_dt = ref_times[1] - ref_times[0]
        self.duration = ref_times[-1]
        self.n_ref_frames = len(ref_times)
        
        self.sim_dt = self.model.opt.timestep
        self.ctrl_dt = ctrl_dt
        self.sim_steps_per_ctrl = max(1, int(ctrl_dt / self.sim_dt))
        
        self.kp = np.full(self.model.nu, kp)
        self.kd = np.full(self.model.nu, kd)
        self.action_scale = action_scale
        self.force_limits = get_force_limits(self.model)
        
        self.reward_fn = MotionTrackingReward(self.model)
        
        self.act_dim = self.model.nu
        self.phase = 0.0
        self.time = 0.0
    
    def reset(self, start_frame=None):
        """Reset environment with Reference State Initialization."""
        if start_frame is None:
            start_frame = np.random.randint(0, self.n_ref_frames)
        
        self.phase = start_frame / self.n_ref_frames
        self.time = self.ref_times[start_frame]
        self.current_ref_idx = start_frame
        
        self.data.qpos[:] = self.ref_qpos[start_frame]
        self.data.qvel[:] = 0
        mujoco.mj_forward(self.model, self.data)
        
        return self._get_obs()
    
    def _get_obs(self):
        ref_idx = self.current_ref_idx % self.n_ref_frames
        ref_pose = self.ref_qpos[ref_idx]
        
        obs = np.concatenate([
            [self.data.qpos[2]],
            self.data.qpos[3:7],
            self.data.qpos[7:],
            self.data.qvel[:6],
            self.data.qvel[6:],
            ref_pose[7:] - self.data.qpos[7:],
            ref_pose[3:7],
            [np.sin(2 * np.pi * self.phase),
             np.cos(2 * np.pi * self.phase)],
        ])
        return obs
    
    def step(self, action):
        ref_idx = self.current_ref_idx % self.n_ref_frames
        ref_pose = self.ref_qpos[ref_idx]
        
        target_joints = ref_pose[7:] + action * self.action_scale
        
        for _ in range(self.sim_steps_per_ctrl):
            torques = self.kp * (target_joints - self.data.qpos[7:]) - self.kd * self.data.qvel[6:]
            self.data.ctrl[:] = np.clip(torques, -self.force_limits, self.force_limits)
            mujoco.mj_step(self.model, self.data)
        
        self.time += self.ctrl_dt
        self.current_ref_idx += max(1, int(self.ctrl_dt / self.ref_dt))
        self.phase = (self.current_ref_idx % self.n_ref_frames) / self.n_ref_frames
        
        ref_idx_new = self.current_ref_idx % self.n_ref_frames
        ref_qvel = np.zeros(self.model.nv)
        if ref_idx_new + 1 < self.n_ref_frames:
            ref_qvel[6:] = (self.ref_qpos[ref_idx_new + 1, 7:] - self.ref_qpos[ref_idx_new, 7:]) / self.ref_dt
        
        self.ref_data.qpos[:] = self.ref_qpos[ref_idx_new]
        mujoco.mj_forward(self.model, self.ref_data)
        
        reward, reward_info = self.reward_fn.compute(
            self.data, self.ref_qpos[ref_idx_new], ref_qvel, self.ref_data
        )
        
        pelvis_height = self.data.qpos[2]
        done = pelvis_height < 0.3
        
        obs = self._get_obs()
        return obs, reward, done, reward_info

env = MotionTrackingEnv(G1_XML_PATH, ref_qpos, ref_times)
obs = env.reset(start_frame=0)
print(f"Observation dimension: {obs.shape[0]}")
print(f"Action dimension: {env.act_dim}")
print(f"Observation breakdown:")
print(f"  Root height:     1")
print(f"  Root quat:       4")
print(f"  Joint angles:    {env.model.nq - 7}")
print(f"  Root velocity:   6")
print(f"  Joint velocity:  {env.model.nv - 6}")
print(f"  Joint ref error: {env.model.nq - 7}")
print(f"  Ref root quat:   4")
print(f"  Phase (sin,cos): 2")
print(f"  Total:           {1+4+(env.model.nq-7)+6+(env.model.nv-6)+(env.model.nq-7)+4+2}")

In [ ]:
# Run a short episode with random actions to show the environment works
obs = env.reset(start_frame=0)
episode_rewards = []
episode_info = {k: [] for k in ['qpos', 'qvel', 'end_effector', 'com', 'root_orient']}

renderer_env = mujoco.Renderer(env.model, height=480, width=640)
env_frames = []

for step in range(100):
    # Random action (in practice, this would come from the RL policy)
    action = np.random.randn(env.act_dim) * 0.1  # small random perturbations
    
    obs, reward, done, info = env.step(action)
    episode_rewards.append(reward)
    for k in episode_info:
        episode_info[k].append(info[k])
    
    # Render
    renderer_env.update_scene(env.data)
    env_frames.append(renderer_env.render().copy())
    
    if done:
        print(f"Episode terminated at step {step} (humanoid fell)")
        break

renderer_env.close()

print(f"Episode length: {len(episode_rewards)} steps")
print(f"Mean reward: {np.mean(episode_rewards):.4f}")
print(f"Min/Max reward: {np.min(episode_rewards):.4f} / {np.max(episode_rewards):.4f}")

media.show_video(env_frames, fps=25, title='RL env episode with random actions')

In [ ]:
# Run a "zero action" episode — pure reference tracking via PD
obs = env.reset(start_frame=0)
zero_rewards = []
zero_info = {k: [] for k in ['qpos', 'qvel', 'end_effector', 'com', 'root_orient']}
zero_frames = []
renderer_z = mujoco.Renderer(env.model, height=480, width=640)

for step in range(100):
    action = np.zeros(env.act_dim)  # zero residual -> pure reference tracking
    obs, reward, done, info = env.step(action)
    zero_rewards.append(reward)
    for k in zero_info:
        zero_info[k].append(info[k])
    
    renderer_z.update_scene(env.data)
    zero_frames.append(renderer_z.render().copy())
    
    if done:
        print(f"Episode terminated at step {step}")
        break

renderer_z.close()

print(f"Zero-action episode: {len(zero_rewards)} steps, mean reward: {np.mean(zero_rewards):.4f}")

# Compare
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(episode_rewards, label='Random actions', alpha=0.8)
ax.plot(zero_rewards, label='Zero actions (pure PD tracking)', alpha=0.8)
ax.set_xlabel('Step')
ax.set_ylabel('Reward')
ax.set_title('Episode Reward: Random vs Zero Actions')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('imgs/episode_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

media.show_video(zero_frames, fps=25, title='Zero-action episode (pure PD reference tracking)')

## Summary
## 总结

This tutorial covered MuJoCo fundamentals for motion tracking RL using the **Unitree G1** humanoid robot:
本教程以**宇树 G1**人形机器人为例，覆盖了动作跟踪强化学习所需的 MuJoCo 基础：

- Model structure: `model.njnt`, `model.nbody`, `model.nu`.
- 模型结构：`model.njnt`、`model.nbody`、`model.nu`。
- State access: `data.qpos`, `data.qvel`.
- 状态访问：`data.qpos`、`data.qvel`。
- Forward kinematics: `data.xpos`, `data.xquat`, `data.subtree_com`.
- 正向运动学：`data.xpos`、`data.xquat`、`data.subtree_com`。
- Quaternion math: `mju_mulQuat`, `mju_negQuat`, `mju_quat2Vel`, `mju_subQuat`.
- 四元数数学：`mju_mulQuat`、`mju_negQuat`、`mju_quat2Vel`、`mju_subQuat`。
- Actuator control: `data.ctrl`, `model.actuator_gear`.
- 执行器控制：`data.ctrl`、`model.actuator_gear`。
- Reference motion: direct `qpos` setting with `mj_forward`.
- 参考动作：结合 `mj_forward` 的 `qpos` 直接设定。
- Tracking reward: `exp(-k * error^2)` with multiple components.
- 跟踪奖励：由多个分量组成的 `exp(-k * error^2)`。
- Full tracking loop: PD control + reward computation + rendering.
- 完整跟踪循环：PD 控制 + 奖励计算 + 渲染。
- RL environment design: observation / action / episode structure.
- RL 环境设计：观测 / 动作 / 回合结构。

### Next steps toward a full BeyondMimic-style system
### 迈向完整 BeyondMimic 风格系统的下一步

1. RL training: use PPO/SAC in [Stable-Baselines3](https://stable-baselines3.readthedocs.io/) or [RSL-RL](https://github.com/leggedrobotics/rsl_rl).
1. RL 训练：使用 [Stable-Baselines3](https://stable-baselines3.readthedocs.io/) 或 [RSL-RL](https://github.com/leggedrobotics/rsl_rl) 的 PPO/SAC。
2. Real MoCap data: load `.bvh` / `.c3d` files and retarget to the Unitree G1 model.
2. 真实 MoCap 数据：加载 `.bvh` / `.c3d` 文件并重定向到宇树 G1 模型。
3. Domain randomization: randomize dynamics parameters for sim-to-real transfer.
3. 领域随机化：随机化动力学参数以提升 sim-to-real 迁移能力。
4. Adversarial training (AMP): replace hand-crafted rewards with a learned discriminator.
4. 对抗训练（AMP）：用学习得到的判别器替代手工奖励。
5. Diffusion policy distillation: train a diffusion model on successful tracking rollouts.
5. 扩散策略蒸馏：在成功跟踪轨迹上训练扩散模型。

